In [1]:
import os

# Imports para envio de email 
import smtplib
from email.mime.multipart import MIMEMultipart # Cria as menssagens para envio de mensagem
from email.mime.text import MIMEText # Corpo da mensagem
from email.mime.base import MIMEBase # Anexar arquivos de imagem
from email import encoders # Serve para transferir os arquivos como pdf e imagens são dados binarios que ao enviar ele vai direto para o lixo
from email.utils import encode_rfc2231
from email.header import Header # 

from urllib.parse import quote # Prepara texto para ser usado em links

from dotenv import load_dotenv
from datetime import datetime, timedelta, date
import requests # Serve para o script "converse" com sites

In [10]:
# A data 
data_universal = (datetime.now() + timedelta(days=0)).date()
# Se for segunda vai fazer um ajuste
if data_universal.weekday() == 0:
    data_universal -= timedelta(3)
else:
    data_universal -= timedelta(1)

mes = data_universal.strftime('%m')
ano_mes = data_universal.strftime('%Y-%m')
data_corpo_email = data_universal.strftime('%d/%m/%Y')

print(data_universal)
print(mes)
print(ano_mes)
print(data_corpo_email)

2026-04-20
04
2026-04
20/04/2026


In [ ]:
# Variaveis
email_remetente = os.getenv('EMAIL')
senha_email = os.getenv('SENHA_EMAIL')

def enviar_email_relatorio(remetente, senha, destinatarios, cc, assunto, corpo_email, anexo_path, assinatura_path):

    msg = MIMEMultipart()
    msg['From'] = remetente
    msg['To'] = ", ".join(destinatarios)
    msg['Cc'] = ", ".join(cc)
    msg['Subject'] = assunto

    # Corpo do e-mail em HTML para incorporar a imagem e a assinatura 
    html_body = f"""
    <html>
        <body>
            <p>{corpo_email}</p>
            <p>Atenciosamente,</p>
            <img src="cid:assinatura_email" alt="Assinatura">
        </body>
    </html>
    """
    # Anexando o html_body ao mimemultpart
    msg.attach(MIMEText(html_body, 'html'))

    # Anexar a imagem da assinatura 
    try:
        with open(assinatura_path, 'rb') as sig_file:
            signature = MIMEBase('image', 'png') # Ajusta 'png' conforme formato da assinatura
            signature.set_payload(sig_file.read()) # Vai ler e carregar a imagem
            encoders.encode_base64(signature) # Criptografia em texto base64
            signature.add_header('Content-Disposition', 'inline', filename=os.path.basename(assinatura_path))
            signature.add_header('Content-ID', '<assinatura_email>')
            # Anexa a mensagem principal
            msg.attach(signature) 

    except FileNotFoundError:
        print(f'Deu ruim com o arquivo {assinatura_path}')
        return False

    except Exception as e:
        print(f"Erro ao anexar imagem da assinatura {e}")
        return False
    
    # Anexa arquivos excel
    arquivos_anexados_com_sucesso = 0
    for caminho in anexo_path:
        try:
            if os.path.exists(caminho):
                nome_arquivo = os.path.basename(caminho)
                with open(caminho, 'rb') as anexo_file:
                    part = MIMEBase('application', 'octet-straem') # Sequencia de bytes desconhecida 
                    part.set_payload(anexo_file.read()) # Vai ler e carregar arquivos
                    encoders.encode_base64(part)
                    # Ler com caractere especial
                    part.add_header('Content-Disposition', 'attachment', filename=Header(nome_arquivo, 'utf-8').encode()) # Extraio o nome do arquivo e coloco aqui

                    msg.attach(part)

                    # Soma 1 cada vez que o arquivo entrar
                    arquivos_anexados_com_sucesso += 1
                    print(f"Arquivo anexado: {nome_arquivo}")
            else:
                print(f"Erro: Arquivo não encontrado {caminho}")
        except Exception as e:
            print(f"Erro ao anexar arquivo: {e}")
            return False
    
    if arquivos_anexados_com_sucesso == 0:
        print(f'O envio para {destinatarios} foi abortado')
        return False
    
    # FINAL
    try:
        # Configuração do servidor SMTP (simple mail transfer protocol)
        with smtplib.SMTP('smtp.office365.com', 587) as server: # Liga com o servidor o outlook na porta padrão para envios
            server.starttls() # para o envio ir criptografado (é necessario)
            server.login(remetente, senha)

            todos_os_destinatarios = destinatarios + cc

            conteudo_email = msg.as_string() # Tudo em padrão texto seguindo o padrão MIME

            server.sendmail(remetente, todos_os_destinatarios, conteudo_email) # Envio
            server.quit()
    except Exception as e:
        print(f'Erro ao enviar email {e}')
        return False


In [ ]:
# Destinatarios
emails_para_enviar = [
    {
        # Controle de Evolução
        "DESTINATARIOS" : [""],
        "CC" : [""],
        "ASSUNTO_EMAIL" : f"Teste de envio 01 - {data_corpo_email}",
        "CORPO_EMAIL_TEXTO" : "Olá, bom dia. <br>Segue<br>",
        "ARQUIVOS" : [
            f"",
            f""

        ]
    }
]

In [28]:
# Obtendo o diretório atual do script
DIRETORIO_EXCEL = fr""
DIRETORIO_EMAIL = os.getcwd()
CAMINHO_ASSINATURA = os.path.join(DIRETORIO_EMAIL, "assinatura.png")

In [29]:
# Chamada da função para enviar o e-mail
if __name__ == "__main__":
    for item in emails_para_enviar:
        lista_atual_anexos = []

        # Lista inteira de anexos
        for nome_arquivo in item["ARQUIVOS"]:
            caminho_completo = os.path.join(DIRETORIO_EXCEL, nome_arquivo)
            lista_atual_anexos.append(caminho_completo)

        # Agora (fora do loop) envia o email
        print(f"Preparando para envio de email: {item['ASSUNTO_EMAIL']}")

        sucesso = enviar_email_relatorio(
            remetente=email_remetente,
            senha=senha_email,
            destinatarios=item["DESTINATARIOS"],
            cc=item["CC"],
            assunto=item["ASSUNTO_EMAIL"],
            corpo_email=item["CORPO_EMAIL_TEXTO"],
            anexo_path=lista_atual_anexos,
            assinatura_path=CAMINHO_ASSINATURA
        )
        if sucesso:
            print(f"E-mail enviado com sucesso para {item['ASSUNTO_EMAIL']}")
        else:
            print(f"E-email com falha ao enviar para {item['ASSUNTO_EMAIL']}")

Preparando para envio de email: Teste de envio 01 - 20/04/2026
Erro ao enviar email (535, b'5.7.3 Authentication unsuccessful [CP3P284CA0135.BRAP284.PROD.OUTLOOK.COM 2026-04-22T00:01:37.634Z 08DE9DD0F34607CC]')
E-email com falha ao enviar para Teste de envio 01 - 20/04/2026
